# 第30课 · 把分数变成信念 — Softmax 与交叉熵（cross entropy），手推梯度 p − y

**目标**：把模型输出的分数 `logits` 转成概率，再用交叉熵计算正确类别对应的损失。

**为什么对 Aurora 重要**：Month 2 关键词分类器（Classifier）和 Month 3 ASR 的输出层都调用 `softmax`，训练循环里的 `cross_entropy` 就是指导梯度更新的那把尺子。


← **上一课**　[L29 · 常见概率分布](L29_distributions.ipynb)

> 上节课学习了 **常见概率分布**：均匀、高斯、伯努利：PDF / CDF 与采样。  
> 本课将探讨 **Softmax 与交叉熵**。

## 学习目标

完成本课后你应该能够：

1. 实现数值稳定的 `softmax(z)`，把任意 logits 向量变为和为 1 的概率分布
2. 实现 `cross_entropy(probs, true_idx)` = `-log(probs[true_idx])`，理解为什么只看真实类别那一格
3. 解释 H(p,q) = H(p) + D_KL(p‖q) 中各项含义，以及 `log₂`（bits）与 `ln`（nats）的差异
4. 手推并数值验证梯度公式 `dL/dz_i = p_i − y_i`


## 本课剧情：模型给出了分数，但我们要的是概率

你训练了一个关键词识别器，最后一层输出三个数字：

```
logits = [2.0, 1.0, 0.1]  ← 对应 speech / music / noise
```

这些数字不是概率——它们可以是负数、不加和为 1、没有明确含义。

**softmax** 把 logits 变成概率分布：

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

结果：[0.659, 0.242, 0.099]，加和=1，每个值在 (0,1) 之间。

**交叉熵（cross entropy，CE）** 衡量这个概率分布离"理想答案"有多远：

$$\text{CE} = -\log(p_{\text{true}})$$

若真实类别是 speech（idx=0），则 CE = -log(0.659) ≈ 0.417。  
模型越自信（p[true]→1），loss 越小（-log(1)=0）。

为什么这两个函数如此重要？  
几乎所有分类神经网络的最后两行都是：`probs = softmax(logits)` + `loss = cross_entropy(probs, target)`。  
本课手推梯度：`dL/dz_i = p_i - y_i`（softmax 概率减 one-hot 标签）——这是反向传播到最后一层的公式。

## 🤔 为什么工程师要发明它？(Why did engineers invent this?)

- **不用它会怎样？** 模型最后一层吐出的是 `[2.0, 1.0, 0.1]` 这种任意实数（logits），可能为负、加和不为 1，没法直接当「我有多确定」来读，也没法跟标签比对算损失。
- **它解决了什么真实问题？** softmax 把这排分数压成一个合法概率分布（每项 ∈ (0,1)、加和为 1）；交叉熵再把「预测分布离正确答案有多远」变成一个**可求导的标量损失**，训练才有梯度可跟。
- **后面哪里还会再用到？** 几乎每个分类网络的最后两行都是它——关键词识别、**L63**（KWS 模型）、ASR 解码、LLM 预测下一个 token。

## ⚠️ 常见误解

> 不要把 softmax 当成「把数字归一化的小技巧」。它其实是把 logits 变成一个**可求导的概率分布**——正因为它光滑可导，交叉熵损失才能一路反传到每个 logit，这就是本课末尾要手推的 `dL/dz_i = p_i − y_i`。

## 1. logits：模型先给分数，不直接给概率

分类器最后一层常输出一组任意实数，叫 `logits`。它们可以是负数，也不需要相加等于 1。softmax 的任务是把这组分数变成概率分布。

**为什么 softmax 要先减最大值**：`np.exp(z)` 在 z > 709 时溢出为 `inf`。减去 `max(z)` 后，最大那个指数变成 `exp(0) = 1`，其余更小，不会溢出。这个变换不影响结果——分子分母同除以 `exp(max(z))`，概率不变。

**为什么 cross_entropy 只看真实类别**：训练目标是最大化模型给出正确类别的概率，即最大化 `log P(correct)`，等价于最小化 `-log P(correct)`。其余类别的概率，softmax 的归一化（Normalization）已经顺带把它们压低了，不用再单独写进损失里。

### 🔍 为什么交叉熵只看真实类别那一项？ — 极大似然的视角

你可能会问："如果我想让模型更自信，为什么不把**所有**类别的概率都写进损失里，比如惩罚错误类别的高概率？"

**答案**：Softmax 的约束（所有概率加和 = 1）已经自动处理了。

想象分类问题的概率模型：给定一个音频 x，真实类别是 t（比如 speech），模型给出的概率分布是 `q(y|x) = softmax(z)`。

极大似然（Maximum Likelihood）的原理说：最好的参数是那些**最大化真实标签概率**的参数，即最大化 `log q(t|x)`。

```
最大化 log(q_t)  ≡  最小化 -log(q_t)
     ↑                    ↑
真实类别概率高  交叉熵损失(只看 q_t 这一项)
```

因为 softmax 有约束 `Σ q_y = 1`，所以：
- 当你最大化 `q_t`（正确类别概率）时
- 其他类别的概率自动被 softmax 的"分数饼"约束压低了——因为总和固定为 1

这是条件概率分布的天然性质，**不需要在损失里单独写出**。正式地讲，这等价于对真实分布 `p(y|x) = [0, ..., 1, ..., 0]`（one-hot）的极大似然估计。

### 📌 One-hot 编码：标签的标准形式

在交叉熵和梯度推导中，"真实标签" 用 **one-hot 向量** 表示：

```
真实类别是第 t 类（比如 t=0）
    ↓
one-hot: y = [1, 0, 0, ...]
            ↑ 真实类别位置为 1
            其余位置为 0
```

**为什么这样编码**？
- 数学上，one-hot y 和概率分布 p（来自 softmax）格式相同（都是向量，元素非负，和为 1）
- 交叉熵可以写成：`H(p, q) = -Σ_i y_i log(q_i)`，虽然 y 里只有一个 1，所以只有真实类别项有贡献
- 梯度公式 `dL/dz = p - y` 才能统一表达（后面会推导）

代码中常见的one-hot转换：
```python
true_idx = 0           # 真实类别的下标
y = np.zeros(3)        # 三类问题
y[true_idx] = 1.0      # 对应位置置 1
# 结果：y = [1.0, 0.0, 0.0]
```

In [ ]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
print('logits =', logits)
print('sum(logits) =', logits.sum(), ' 不是概率和')
print('argmax =', int(np.argmax(logits)), ' 分数最高的类别')


## 2. 数值稳定的 softmax

**为什么要"数值稳定"？**

直接计算 `exp(z)` 时，如果 z 中有很大的值（如 z=1000），`exp(1000)` 会溢出为 `inf`。

**数学上等价的技巧**：先减去最大值 c = max(z)，再计算 exp：

$$\text{softmax}(z)_i = \frac{e^{z_i - c}}{\sum_j e^{z_j - c}}$$

这不改变结果（分子分母同乘 e^{-c}），但把最大指数变为 0，避免溢出。

**手算验证**（z = [2, 1, 0.1]，c = 2）：

```
exp([0, -1, -1.9]) = [1.000, 0.368, 0.150]
sum = 1.518
softmax = [0.659, 0.242, 0.099]   ← 和 = 1 ✓
```

**关键性质**：
- 若所有 logits 相等，softmax 输出均匀分布：[1/n, 1/n, ..., 1/n]
- logits 差距越大，概率越集中（最高类趋近1，其余趋近0）
- 梯度：`dL/dz_i = p_i - y_i`（p=softmax，y=one-hot）

> **实现提示**：`z_stable = z - z.max(); exp_z = np.exp(z_stable); return exp_z / exp_z.sum()`

### 💭 为什么 softmax 的"分数饼"会自动压低错误类别？ — 指数函数的性质

一个学生可能会问：我只优化"正确类别的概率"，其他类别的概率怎么保证会被压低？

**答案来自指数函数的一个关键性质**：**exp() 把差异放大**。

**具体例子**：
- logits = [2.0, 1.0, 0.1]（三类）
- 如果不用 exp()，只是简单归一化（比如除以总和），[2.0, 1.0, 0.1] / 3.1 = [0.645, 0.323, 0.032]
- 但用 softmax（先 exp 后归一化）：
  - exp([2.0, 1.0, 0.1]) = [7.39, 2.72, 1.11]
  - softmax = [0.659, 0.242, 0.099]

看起来没什么不同？再看一个例子，当分数差异**更大**时：

| logits | 简单除法 | Softmax 概率 |
|--------|---------|------------|
| [5.0, 1.0, 0.1] | [0.780, 0.196, 0.024] | [0.989, 0.009, 0.002] |
| [10.0, 1.0, 0.1] | [0.830, 0.083, 0.008] | [0.9999, 0.00005, 0.00001] |

**关键观察**：exp() 把分数差异**非线性放大**。最高的分数对应的概率更高，最低的更低。这就是为什么训练时只优化 `p_t`（正确类别）就足够了——softmax 的指数形式已经自动"惩罚"了错误类别。

**数学直觉**：softmax 的分子是 $e^{z_i}$，分母是 $\sum_j e^{z_j}$。当你增加 $z_t$（正确类别的分数）时：
- 分子 $e^{z_t}$ 指数增长
- 其他 $e^{z_j}$（$j \neq t$）相对于分母的比例下降
- 约束条件 $\sum p = 1$ 自动把这个增长转化为对其他类别概率的压低



## 实验入口：概率量如何从样本（Sample）里出现

固定一组 logits，观察 softmax 输出的概率向量如何随 logits 差距的扩大而集中；再固定 target，看 `cross_entropy` 的数值如何随正确类别的概率反向变化。


### 写代码前，先把变量表补完整

写 `softmax` 前明确三件事：
- 输入：`z`，任意实数向量（logits），形状 `(n_classes,)`
- 关键步骤：减去 `max(z)` 防溢出 → 对 `exp(z)` 结果归一化
- 返回：概率向量，各元素 ≥ 0，和恰好为 1


In [ ]:
import numpy as np
def softmax(z):
    z = np.asarray(z, dtype=float)
    # ✏️ TODO: 返回概率分布(和为1)
    raise NotImplementedError("请实现 softmax：减去 max(z)，对 exp(z) 归一化后返回概率向量")


### 🔧 Numpy 快速参考：这里会用到的函数

**`np.asarray(z, dtype=float)`**：
- 作用：把输入转成 NumPy 数组，并确保数据类型是浮点数
- 为什么需要？如果输入是列表、元组或整数，直接用 `np.exp()` 可能有类型问题。`dtype=float` 保证之后的对数、除法操作不会丢失精度。
- 例：`np.asarray([2, 1, 0.1], dtype=float)` → `array([2. , 1. , 0.1])`

**`np.log(x)` vs `np.log2(x)`**：
- `np.log(x)`：自然对数，即 ln(x)（以 e ≈ 2.718 为底）
- `np.log2(x)`：以 2 为底的对数（单位是 bits）
- 在代码中哪个用？在这节课里，交叉熵损失函数通常用 `np.log()`（自然对数）；Shannon 熵的计算如果想要 bits 单位则用对数换底：`np.log(p) / np.log(2)` 等价于 `np.log2(p)`



## 动手观察：分数差距如何改变概率

保持类别顺序不变，只放大 logits 的差距。分数差距越大，softmax 后的概率越集中。


In [ ]:
import numpy as np

base = np.array([2.0, 1.0, 0.1])
for scale in [0.5, 1.0, 2.0, 5.0]:
    z = base * scale
    e = np.exp(z - np.max(z))
    p = e / e.sum()
    print(f'scale={scale:>3}: logits={np.round(z,2)} -> probs={np.round(p,3)}')


## 代码实验：同一个正确类别，不同置信度

交叉熵只读取真实类别那一格的概率。正确类别概率越高，loss 越小。


In [ ]:
try:
    # 运行前先完成上面的 softmax(z)
    for z in [[2.0, 1.0, 0.1], [5.0, 1.0, 0.1], [1.1, 1.0, 0.9]]:
        p = softmax(z)
        print('logits=', z, 'probs=', np.round(p, 3), 'true-class prob=', round(float(p[0]), 3))
except (NotImplementedError, TypeError) as _e:
    print('⬜ 请先实现函数后再运行此单元格：', _e)


In [ ]:
p = softmax([2.0, 1.0, 0.1])
print('概率:', np.round(p, 3), ' 和 =', round(float(p.sum()), 6))
assert abs(p.sum() - 1.0) < 1e-9, '概率应和为 1'
assert np.argmax(p) == 0, '最大分数应得最大概率'
print('✅ softmax 通过。')

## 3. 信息熵（Shannon Entropy）— 不确定性的度量

**定义**：对离散概率分布 p = (p₁, ..., pₙ)，Shannon 熵（Shannon Entropy）为

> H(p) = −Σᵢ pᵢ log₂ pᵢ

（约定 0·log 0 = 0；以 log₂ 为底时单位为 bit，以 ln 为底时单位为 nat）

**三条关键性质**：

| 性质 | 表达式 | 直觉 |
|------|--------|------|
| 非负性 | H(p) ≥ 0 | 信息量不为负 |
| 最大值 | H(均匀分布) = log₂ n | 什么都不知道时不确定性最高 |
| 确定性 | H(确定分布) = 0 | 完全知道结果时不需要任何信息 |

**与交叉熵的关系**：

```
H(p, q) = H(p) + D_KL(p‖q)
交叉熵  = 自熵 + KL散度（预测分布 q 与真实分布 p 的距离）
```

**为什么分类训练用交叉熵等价于最小化 KL 散度**：  
真实标签 p 在训练集上固定 → H(p) 是常数 → 最小化 H(p,q) = 最小化 D_KL(p‖q)。  
这解释了 cross_entropy loss 能驱动模型预测分布逼近真实分布的数学原因。

**Aurora 连接**：音频事件的真实标签不确定性越高（H(p) 越大），同一批数据能提供的"有效信息量"越少，模型需要更多样本才能收敛。

### 📊 KL 散度：两个概率分布的"距离"

在讲 Shannon 熵的三条性质之前，我们需要理解 **KL 散度**（Kullback-Leibler Divergence），因为它将直接说明"为什么交叉熵等价于最小化 KL 散度"。

**定义**：给定两个概率分布 p 和 q（同一个样本空间），KL 散度定义为

$$D_{\text{KL}}(p \| q) = \sum_i p_i \log \frac{p_i}{q_i} = \sum_i p_i \log(p_i) - \sum_i p_i \log(q_i)$$

**直觉**：KL 散度衡量用分布 q 来"近似"分布 p 的代价。它不是对称的——`D_KL(p||q)` 和 `D_KL(q||p)` 通常不相等。

**三条关键性质**：

| 性质 | 表达式 | 含义 |
|------|--------|------|
| 非负性 | D_KL(p\|\|q) ≥ 0 | 距离不为负 |
| 相等条件 | D_KL(p\|\|q) = 0 ⟺ p = q | 两分布相同时距离为零 |
| 渐近行为 | q_i → 0 but p_i ≠ 0 ⟹ D_KL → ∞ | 如果 q 认为不可能的事件 p 认为可能，距离无穷大 |

**KL 散度与交叉熵的关系**：

观察 KL 散度的展开式：

$$D_{\text{KL}}(p \| q) = \sum_i p_i \log(p_i) - \sum_i p_i \log(q_i)$$

第一项 `Σ p_i log(p_i)` 就是**自熵 H(p)**（定义见下）；第二项的负数 `-Σ p_i log(q_i)` 就是**交叉熵 H(p, q)**。所以：

$$H(p, q) = H(p) + D_{\text{KL}}(p \| q)$$

**为什么这对分类训练很关键**？
- 真实标签分布 p（one-hot，比如 [1,0,0]）在训练集上**固定不变**
- 因此 H(p) 是常数
- 最小化交叉熵 ≡ 最小化 D_KL(p||q)
- 这说明：训练过程本质上是让模型的预测分布 q 逐步逼近真实分布 p



In [ ]:
import numpy as np

def shannon_entropy(p, base=2):
    """
    H(p) = -Σ p_i log_b(p_i)，约定 0·log(0) = 0
    
    参数：
      p: 概率分布（向量，和为 1）
      base: 对数底数。base=2 时单位为 bits，base=e（≈2.718）时单位为 nats
    
    关键步骤：
      1. p = p[p > 0]：跳过零概率项（因为它们对熵的贡献为 0）
      2. np.log(p)：自然对数（ln）
      3. np.log(base)：对数换底 — log_b(p_i) = ln(p_i) / ln(b)
    """
    p = np.asarray(p, dtype=float)
    p = p[p > 0]          # 跳过 p_i=0 的项（0·log0 → 0）
    # 对数换底公式：log_b(x) = ln(x) / ln(b)
    # np.log 是自然对数 ln；如果直接用 np.log2(p) 则得 log₂(p)
    return float(-np.sum(p * np.log(p)) / np.log(base))

# ── 性质 1 & 3：均匀分布熵最大；确定分布熵为 0 ──────────────────────────
n       = 4
uniform = np.ones(n) / n
peaked  = np.array([0.97, 0.01, 0.01, 0.01])
certain = np.array([1.0, 0.0, 0.0, 0.0])

print('熵（bits）对比：')
print(f'  均匀分布  H = {shannon_entropy(uniform):.4f}  (理论上界 log₂{n} = {np.log2(n):.4f})')
print(f'  集中分布  H = {shannon_entropy(peaked):.4f}')
print(f'  确定分布  H = {shannon_entropy(certain):.4f}  ← 无不确定性')

# ── 性质 2：H(p,q) ≥ H(p)，等号当且仅当 p=q ────────────────────────────
def cross_entropy_h(p, q):
    """H(p,q) = -Σ p_i log₂ q_i（以 2 为底，单位 bits）"""
    p, q = np.asarray(p, float), np.clip(np.asarray(q, float), 1e-15, None)
    return float(-np.sum(p * np.log(q)) / np.log(2))

p_true = np.array([0.7, 0.2, 0.1])
q_good = np.array([0.65, 0.25, 0.10])   # 接近真实
q_bad  = np.array([0.10, 0.10, 0.80])   # 远离真实

Hp       = shannon_entropy(p_true)
Hpq_good = cross_entropy_h(p_true, q_good)
Hpq_bad  = cross_entropy_h(p_true, q_bad)

print(f'\n自熵 H(p)        = {Hp:.4f} bits  ← 损失的理论下界（D_KL=0 时才能达到）')
print(f'H(p, q_好预测)  = {Hpq_good:.4f} bits  KL = {Hpq_good - Hp:.4f}')
print(f'H(p, q_差预测)  = {Hpq_bad:.4f} bits  KL = {Hpq_bad - Hp:.4f}')
print('→ 训练目标是压缩 KL；因 H(p) 固定，最小化交叉熵 ≡ 最小化 KL 散度')

assert shannon_entropy(certain) < 1e-9,          '确定分布熵应为 0'
assert abs(shannon_entropy(uniform) - np.log2(n)) < 1e-9, '均匀分布熵应为 log₂(n)'
assert Hpq_good >= Hp - 1e-9, 'H(p,q) ≥ H(p) 下界性质验证失败'
assert Hpq_bad  >= Hp - 1e-9, 'H(p,q) ≥ H(p) 下界性质验证失败'
print('✅ Shannon 熵三条性质全部验证通过')

### 💡 为什么 Shannon 熵是这个样子？— 信息论的故事

给你一个问题：**你需要用多少个二分法问题来猜出一个概率分布的结果**？

例如，掷均匀硬币（p=[0.5, 0.5]），你需要问 **1 个** 二分法问题才能确定是正面还是反面。  
如果掷四面骰子（p=[0.25, 0.25, 0.25, 0.25]），你需要问 **2 个** 二分法问题。  
但如果骰子很偏（p=[0.99, 0.01/3, 0.01/3, 0.01/3]），结果总是第一面，**0 个问题**基本就够了。

**Shannon 1948 年的洞察**：给定分布 p，平均需要的"二分法问题数"就是

$$H(p) = \sum_i p_i \log_2 \frac{1}{p_i} = -\sum_i p_i \log_2 p_i \quad \text{（单位：bits）}$$

这就是 Shannon 熵。为什么是 `-log p_i`？因为**不确定性与 p_i 的倒数成反比**：
- p_i 很大（接近 1）→ `log(1/p_i)` 很小 → 这个事件不给你新信息
- p_i 很小（接近 0）→ `log(1/p_i)` 很大 → 这个事件发生时给你很多惊喜

对整个分布取期望（乘以 p_i），得到**平均信息量**，就是 Shannon 熵。

**"约定 0·log 0 = 0" 的数学原因**：

在概率为 0 的事件上，即使 `log(1/p) = ∞`，但因为 p=0，它们对期望值 Σ p_i log(1/p_i) 的贡献是：

$$\lim_{p \to 0^+} p \log(1/p) = \lim_{p \to 0^+} -p \log(p) = 0$$

（用洛必达法则可证）

所以在代码中，我们可以直接跳过 p_i=0 的项，它们的贡献为 0。

**Shannon 熵的三条性质**（现在有了信息论的背景）：

| 性质 | 表达式 | 含义 |
|------|--------|------|
| 非负性 | H(p) ≥ 0 | 信息量不为负 |
| 最大值 | H(均匀分布) = log₂ n | 完全不知道结果时，不确定性最大 |
| 确定性 | H(确定分布) = 0 | 完全知道结果时，不需要任何新信息 |



## 4. ✏️ 实现交叉熵（Cross Entropy） `cross_entropy(probs, true_idx)`

只看真实类别的概率：`loss = −log(probs[true_idx])`。预测越自信且正确，损失越小。

**推理路线**：
1. `probs` 是 softmax 的输出，所有元素 ≥ 0 且和为 1；`true_idx` 是正确类别的下标。
2. 用 `probs[true_idx]` 取出正确类别的概率 p。p 越接近 1 代表模型越自信且正确。
3. 取 `-log(p)`：当 p=1 时损失为 0（完美），p=0.1 时损失约为 2.3，p→0 时损失→∞。一行代码就够。

**参考输入输出**：`probs=[0.665, 0.245, 0.090]`，`true_idx=0` → `loss = -log(0.665) ≈ 0.408`；若 `true_idx=2` → `loss = -log(0.090) ≈ 2.408`。

> 注：这里的 `log` 指自然对数 `ln`；前面 Shannon 熵部分使用 `log₂`，两者只差一个常数因子 `ln(2)`。

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def cross_entropy(probs, true_idx):
    # ✏️ TODO: 返回交叉熵损失
    raise NotImplementedError("请实现 cross_entropy：返回 -np.log(probs[true_idx])")


In [ ]:
try:
    p = softmax([2.0, 1.0, 0.1])
    good = cross_entropy(p, 0)   # 真实类=分数最高的那个 -> 损失小
    bad  = cross_entropy(p, 2)   # 真实类=分数最低的那个 -> 损失大
    print('预测对时损失:', round(good,3), ' 预测错时损失:', round(bad,3))
    assert good < bad
    print('✅ 通过：你做出了分类训练的损失函数。')
except (NotImplementedError, TypeError) as _e:
    print('⬜ 请先实现函数后再运行此单元格：', _e)


## 5. Toy classifier：从一行分数到一行 loss

下面把一条样本的三类输出完整跑一遍。真实类别是 `target=0`，所以 loss 只看 softmax 后第 0 类的概率。


In [ ]:
try:
    class_names = ['speech', 'music', 'noise']
    logits = np.array([3.2, 1.4, -0.5])
    target = 0
    
    probs = softmax(logits)
    loss = cross_entropy(probs, target)
    
    for name, z, p in zip(class_names, logits, probs):
        print(f'{name:>6}: logit={z:+.2f}  prob={p:.3f}')
    print('target =', class_names[target], 'loss =', round(float(loss), 3))
except (NotImplementedError, TypeError) as _e:
    print('⬜ 请先实现函数后再运行此单元格：', _e)


**🔗 Aurora 连接**：Month 2 关键词分类器（目标路径：`src/aurora/speech/`，Month 2 待建）的输出层直接调用 `softmax(z)`，训练循环里的 `cross_entropy(probs, target)` 产出标量损失，梯度从这个标量往回传。Month 3 ASR 在 token 级别逐帧重复相同流程，vocabulary 规模可达数千类，但 `softmax` 和 `cross_entropy` 的逻辑与此处完全相同。

In [ ]:
try:
    for scale in [0.5, 1.0, 2.0, 5.0]:
        z = np.array([2.0, 1.0, 0.1]) * scale
        p = softmax(z)
        loss = cross_entropy(p, 0)
        print(f'scale={scale:>3}: p_true={p[0]:.3f}, loss={loss:.3f}')
except (NotImplementedError, TypeError) as _e:
    print('⬜ 请先实现函数后再运行此单元格：', _e)


## 参数实验：只改一个旋钮

把 `true_idx=0` 对应的 logit 从 2 逐步增大到 10，观察 softmax 概率从 0.659 趋近 1.0，cross_entropy 从 0.417 趋近 0——正确类别分数越高，模型越自信，损失越低。

再把所有 logit 同时加 100（`z + 100`），确认 softmax 输出与未加之前完全相同。这正是「减 max」的意义：绝对数值不影响概率分布，只有类别之间的相对差距才决定概率大小。

## 6. 手推梯度：dL/dz_i = p_i - y_i

设 `z` 为 logits，`p = softmax(z)`，`y` 为 one-hot 标签（真实类别 t 处为 1，其余为 0）。
交叉熵损失：`L = -log(p_t)`。

**推导**（对 logit z_i 求偏导）：

```
L = -log(p_t) = -log(exp(z_t) / sum_j exp(z_j))
  = -z_t + log(sum_j exp(z_j))

dL/dz_i = d/dz_i [-z_t + log(sum_j exp(z_j))]

当 i = t:  dL/dz_t = -1 + exp(z_t)/sum = -1 + p_t = p_t - 1 = p_t - y_t
当 i != t: dL/dz_i =  0 + exp(z_i)/sum =      p_i = p_i - 0 = p_i - y_i

统一写成：dL/dz_i = p_i - y_i
```

这个梯度的直觉：**预测概率与真实标签的差**就是 logit 的梯度。
预测越接近真实标签（`p_t → 1`），梯度 `p_t - 1 → 0`，更新越小。

**Aurora 连接**：Month 2 关键词分类器的 `backward()` 从 `cross_entropy` 开始，第一步计算的正是 `p - y`，它传入最后一层的反向传播。

In [ ]:
# 参考实现（_softmax_ref）：只用于梯度验证，不影响你在 cell 上方实现的 softmax
import numpy as np

def _softmax_ref(z):  # 参考实现，与学生 softmax 不冲突
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()

def ce_gradient(z, true_idx):
    p = _softmax_ref(z)
    y = np.zeros_like(p)
    y[true_idx] = 1.0
    return p - y  # dL/dz = p - y

# 验证：数值梯度 vs 解析梯度
def cross_entropy_loss(z, true_idx):
    p = _softmax_ref(z)
    return -np.log(_softmax_ref(z)[true_idx] + 1e-15)

z = np.array([3.2, 1.4, -0.5])
true_idx = 0
h = 1e-5

analytic_grad = ce_gradient(z, true_idx)
numerical_grad = np.array([
    (cross_entropy_loss(z + h*(np.arange(len(z))==i), true_idx) -
     cross_entropy_loss(z - h*(np.arange(len(z))==i), true_idx)) / (2*h)
    for i in range(len(z))
])

print('解析梯度 (p - y):', np.round(analytic_grad, 6))
print('数值梯度:        ', np.round(numerical_grad, 6))
print('误差:            ', np.round(np.abs(analytic_grad - numerical_grad), 8))

assert np.allclose(analytic_grad, numerical_grad, atol=1e-6)
print('✅ dL/dz_i = p_i - y_i 验证通过：解析梯度与数值梯度吻合')


### 📐 详细推导：d/dz_i log(Σ_j exp(z_j))

你会发现梯度推导涉及链式法则和对复杂表达式求偏导。下面逐步展开，特别适合高中微积分背景的学生。

**第 0 步**：交叉熵损失的原始形式
$$L = -\log(p_t) = -\log\left(\frac{e^{z_t}}{\sum_j e^{z_j}}\right)$$

其中 $z_t$ 是真实类别的 logit，$\sum_j e^{z_j}$ 是所有 logits 的指数和。

**第 1 步**：应用对数的除法规则 $\log(a/b) = \log(a) - \log(b)$
$$L = -\left[\log(e^{z_t}) - \log\left(\sum_j e^{z_j}\right)\right] = -z_t + \log\left(\sum_j e^{z_j}\right)$$

**第 2 步**：对某个 logit $z_i$ 求偏导

现在计算 $\frac{\partial L}{\partial z_i}$。第一项 $-z_t$ 的偏导：
- 如果 $i = t$（正确类别），则 $\frac{\partial (-z_t)}{\partial z_i} = -1$
- 如果 $i \neq t$（错误类别），则 $\frac{\partial (-z_t)}{\partial z_i} = 0$

第二项 $\log(\sum_j e^{z_j})$ 的偏导使用**链式法则**：

$$\frac{\partial}{\partial z_i} \log\left(\sum_j e^{z_j}\right) = \frac{1}{\sum_j e^{z_j}} \cdot \frac{\partial}{\partial z_i}\left(\sum_j e^{z_j}\right)$$

对求和式求偏导（只有 $j=i$ 时有非零项）：
$$\frac{\partial}{\partial z_i}\left(\sum_j e^{z_j}\right) = e^{z_i}$$

所以：
$$\frac{\partial}{\partial z_i} \log\left(\sum_j e^{z_j}\right) = \frac{e^{z_i}}{\sum_j e^{z_j}} = p_i$$

这里 $p_i$ 就是 softmax 的第 i 个分量。

**第 3 步**：合并两项
$$\frac{\partial L}{\partial z_i} = -\mathbb{1}_{i=t} + p_i = p_i - \mathbb{1}_{i=t}$$

其中 $\mathbb{1}_{i=t}$ 是指示函数（当 $i=t$ 时为 1，否则为 0）。

**第 4 步**：用 one-hot 向量重新表达

记 one-hot 标签向量为 $y$，其中 $y_i = \mathbb{1}_{i=t}$（真实类别位置为 1，其余为 0）。则：

$$\boxed{\frac{\partial L}{\partial z_i} = p_i - y_i}$$

这就是梯度公式。**直觉**：
- 当预测正确（$p_t \approx 1$）时，梯度 $p_t - y_t = 1 - 1 = 0$，不需要更新
- 当预测错误（$p_t \ll 1$）时，梯度的绝对值很大，参数会被大幅调整



## 本课收束

现在能用 `softmax(z)` 把任意 logits 向量转成概率分布，用 `cross_entropy(probs, target)` 得到一个标量损失。两个函数合在一起就是分类器「分数 → 概率 → 损失」的完整流程。Month 2 的关键词分类器会直接复用这套代码，Month 3 的 ASR 在 token 级别按相同方式计算训练损失。

**下一课 L31** 会把 softmax、交叉熵与 Shannon 熵画成图；读完后进入 Audio DSP（L32）前，建议先完成 L31 末尾的模块切换清单。

交叉熵的梯度不会停留在公式里：L58 的训练循环会把 `dL/dz = p − y` 真正接到参数更新上；更远处的序列损失（如 L69 的 CTC）只是同一思想的另一种视角。

In [ ]:
try:
    # 小检查：softmax + 交叉熵的数学规律
    import numpy as np
    
    # 规律1：logit 差越大，最高概率越集中
    for scale in [0.1, 1.0, 5.0, 20.0]:
        z = np.array([2.0, 1.0, 0.0]) * scale
        p = softmax(z)
        loss = cross_entropy(p, 0)  # 目标类别=0（最高分）
        if p is None or loss is None:
            print('⬜ 请先实现 softmax 和 cross_entropy')
            break
        print(f'scale={scale:5.1f}  max_prob={p[0]:.4f}  loss={loss:.4f}')
    print('→ scale 越大，模型越自信，交叉熵损失越小。')
except (NotImplementedError, TypeError) as _e:
    print('⬜ 请先实现函数后再运行此单元格：', _e)


---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：softmax + 交叉熵手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：z = [2.0, 1.0, 0.1]，手算数值稳定 softmax：
- c = max(z) = ?
- exp(z - c) = ?（三个值）
- softmax(z) = ?（三个值，加和应=1）

**问 2**：若真实类别是第 0 类（speech），cross_entropy = -log(p[0]) = ?

**问 3**：若 z = [5, 5, 5]（三类分数相等），softmax 输出是什么？  
交叉熵是 -log(?) ≈ ?

**问 4**：softmax 对 logits 的梯度公式：`dL/dz_i = ?`  
（提示：one-hot 标签 y，softmax 概率 p）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：softmax([2,1,0.1])
z = np.array([2.0, 1.0, 0.1])
c = z.max()   # = 2.0
exp_z = np.exp(z - c)
sm_expected = exp_z / exp_z.sum()
assert np.isclose(sm_expected.sum(), 1.0, atol=1e-12)
assert np.isclose(sm_expected[0], 0.6590011, atol=1e-5)
try:
    sm_computed = softmax(z)
    assert np.allclose(sm_computed, sm_expected, atol=1e-10)
    print(f"Q1 ✅  softmax([2,1,0.1])={np.round(sm_computed,4)}，和={sm_computed.sum():.6f}")
except (NotImplementedError, TypeError):
    print("⬜ Q1：请先实现 softmax()，再运行对答案格")

# 问2：CE = -log(p[0]) ≈ 0.4174
ce_expected = -np.log(sm_expected[0])
assert np.isclose(ce_expected, 0.41703, atol=1e-4)
try:
    ce_computed = cross_entropy(sm_expected, 0)
    assert np.isclose(ce_computed, ce_expected, atol=1e-10)
    print(f"Q2 ✅  CE=-log({sm_expected[0]:.4f})={ce_computed:.5f}")
except (NotImplementedError, TypeError):
    print("⬜ Q2：请先实现 cross_entropy()，再运行对答案格")

# 问3：z=[5,5,5] → softmax=[1/3,1/3,1/3]
z3 = np.array([5.0, 5.0, 5.0])
sm3 = np.exp(z3 - z3.max()) / np.exp(z3 - z3.max()).sum()
assert np.allclose(sm3, [1/3, 1/3, 1/3], atol=1e-12)
ce3 = -np.log(1/3)
print(f"Q3 ✅  z=[5,5,5]→softmax=[1/3,1/3,1/3]，CE=-log(1/3)={ce3:.4f}≈1.099")

# 问4：梯度 dL/dz_i = p_i - y_i（one-hot: y[0]=1, rest=0）
y = np.array([1.0, 0.0, 0.0])
grad_expected = sm_expected - y
assert np.isclose(grad_expected.sum(), 0.0, atol=1e-12)  # 梯度之和=0
print(f"Q4 ✅  dL/dz = p - y = {np.round(grad_expected, 4)}  （和=0，是正常性质）")
print(f"     即 dL/dz_0=p[0]-1 (分类器对正确类别'推高'概率)")
print("\n🎉 Softmax+交叉熵白板挑战通过！分类器输出层公式已内化。")

In [ ]:
# ✏️ 本课自评
l30_review = {
    "softmax_formula_stable":     None,  # 记住数值稳定 softmax（减 max）？True/False
    "softmax_implemented":        None,  # softmax 实现并通过断言？True/False
    "cross_entropy_implemented":  None,  # cross_entropy 实现并通过断言？True/False
    "gradient_formula":           None,  # 记住 dL/dz_i = p_i - y_i？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l30_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l30_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L30 全部通关！进入 L31：概率分布可视化')

---

→ **下一课**　[L31 · 概率分布可视化](L31_visual_probability.ipynb)

> 下节课将学习 **概率分布可视化**：PDF、CDF 与交叉熵损失曲面动态演示。